# NeuroICE — Exploration & RMS Baseline

In [101]:
pip install librosa soundfile matplotlib

Note: you may need to restart the kernel to use updated packages.


"chcp" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import librosa
import numpy as np
from tqdm import tqdm
import os
import librosa
import numpy as np
import pandas as pd
import joblib
import pandas as pd
from pathlib import Path
from pathlib import Path
from tqdm import tqdm
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.model_selection import train_test_split

In [4]:

DATA_FILE = Path(r"C:\Users\wer28\Documents\spb_hack\train\train.csv")
train_m = pd.read_csv(DATA_FILE)

if 'source_session_id' in train_m.columns:
    print('train sessions:', train_m['source_session_id'].nunique())



# print('train chunks:', len(train_m), 'train sessions:', train_m['source_session_id'].nunique())
train_m.head()

,filename,label
0,019d24c8-d009-785a-83d8-e584b80145bf_00.wav,normal
1,019d24c8-d009-785a-83d8-e584b80145bf_01.wav,normal
2,019d24c8-d009-785a-83d8-e584b80145bf_02.wav,normal
3,019d24c8-d009-785a-83d8-e584b80145bf_03.wav,normal
4,019d24c8-d009-785a-83d8-e584b80145bf_04.wav,normal


In [2]:
def extract_features_with_context(window_files: list) -> np.ndarray:
    """
    Извлекает признаки из файла (только текущий, без окна).
    """
    if len(window_files) == 0:
        return np.zeros(29, dtype=np.float32)
    
    # Берем центральный файл из окна
    path = DATA_DIR / window_files[len(window_files)//2]
    return extract_features_from_audio(path)

In [5]:
# Вытаскиваем ID сессии из имени файла (берем всё, что до символа '_')
train_m['session_id'] = train_m['filename'].str.split('_').str[0]

In [6]:
train_m['session_id'].nunique() # количество уникальных сессий

57

In [7]:
len(train_m) # количество строк или чанков

570

In [8]:
train_m['label'].value_counts() # распределение меток 

label
normal    570
Name: count, dtype: int64

In [9]:
train_m.head()

,filename,label,session_id
0,019d24c8-d009-785a-83d8-e584b80145bf_00.wav,normal,019d24c8-d009-785a-83d8-e584b80145bf
1,019d24c8-d009-785a-83d8-e584b80145bf_01.wav,normal,019d24c8-d009-785a-83d8-e584b80145bf
2,019d24c8-d009-785a-83d8-e584b80145bf_02.wav,normal,019d24c8-d009-785a-83d8-e584b80145bf
3,019d24c8-d009-785a-83d8-e584b80145bf_03.wav,normal,019d24c8-d009-785a-83d8-e584b80145bf
4,019d24c8-d009-785a-83d8-e584b80145bf_04.wav,normal,019d24c8-d009-785a-83d8-e584b80145bf


In [10]:
# Прямые пути для Windows (раз CSV лежит там, значит и папка рядом)
BASE_DIR = Path(r"C:\Users\wer28\Documents\spb_hack\train")
DATA_DIR = BASE_DIR / "train"
TRAIN_CSV = BASE_DIR / "train.csv"

# TRAIN_CSV = Path(r"C:\Users\wer28\Documents\spb_hack\train\train.csv")
MODEL_PATH = Path("../model.pkl") # Сохраняем в корень проекта
print(f"Проверка путей:")
print(f"Папка с аудио существует: {DATA_DIR.exists()} ({DATA_DIR})")
print(f"Файл CSV существует: {TRAIN_CSV.exists()} ({TRAIN_CSV})")

Проверка путей:
Папка с аудио существует: True (C:\Users\wer28\Documents\spb_hack\train\train)
Файл CSV существует: True (C:\Users\wer28\Documents\spb_hack\train\train.csv)


In [14]:
def extract_features_from_audio(file_path: Path) -> np.ndarray:
    try:
        # sr частота дискретизации, y - количество семплов
        y, sr = librosa.load(str(file_path), sr=48000, mono=True) # попробовать с другой частотой
        if y.size == 0:
            raise ValueError("Empty audio")
        
        # гармоника апмлитуда на кратных частотах, апмлитуда сигнала которая берется на частотах на кратной частоте 
        y_harmonic, y_percussive = librosa.effects.hpss(y) # след итерацию попробовать без этого
        #тон и перкуссия
        #средний коэффициент гаромоники к перкуссии
        harmonic_ratio = np.mean(np.abs(y_harmonic)) / (np.mean(np.abs(y_percussive)) + 1e-9)
        

        # СПЕКТРАЛЬНЫЕ ПРИЗНАКИ
        # 20 коэффициентов описывают "форму" звука (паспорт звука, MFCC - мел-частотные кепстральные коэффициенты)/ характеризует трембр звука

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20) # поменять коэф на 20 с ним будет дольше но лучше работать
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=16, fmax=24000)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        

        # Плоскость спектра (отлично ловит посторонние свисты)
        flatness = librosa.feature.spectral_flatness(y=y)

        # Спектральный центроид: "яркость" звука (свист/гул)
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

        # Спектральный спад: помогает отличить шум от чистого тона
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)


        # ВРЕМЕННЫЕ ПРИЗНАКИ
        # ZCR: частота пересечения нуля (помогает найти трение/скрежет)
        zcr = librosa.feature.zero_crossing_rate(y)
        # RMS: энергия/громкость
        rms = librosa.feature.rms(y=y)


        feature_vector = np.hstack([
            np.mean(mfcc, axis=1),
            np.mean(mel_db, axis=1),
            np.mean(centroid),
            np.mean(rolloff),
            np.mean(zcr),
            np.mean(flatness),
            np.mean(rms), # также думаю вместо средного использвать моду, может, будет лучший скор
            harmonic_ratio, # попробовать добавить персентиль, с маленький доверительный интервалом
        ])

        return feature_vector.astype(np.float32)
    except Exception:
        return np.zeros(29, dtype=np.float32)

In [12]:
def extract_features(file_path):
    try:
        # Загружаем аудио (22050 Hz — золотая середина для CPU и качества)
        y, sr = librosa.load(str(file_path), sr=22050)
        
        # СПЕКТРАЛЬНЫЕ ПРИЗНАКИ
        # MFCC: 20 коэффициентов описывают "форму" звука
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        # Спектральный центроид: "яркость" звука (свист/гул)
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        # Спектральный спад: помогает отличить шум от чистого тона
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        # Плоскость спектра (отлично ловит посторонние свисты)
        flatness = librosa.feature.spectral_flatness(y=y)

        # ВРЕМЕННЫЕ ПРИЗНАКИ
        # RMS: энергия/громкость
        rms = librosa.feature.rms(y=y)
        # ZCR: частота пересечения нуля (помогает найти трение/скрежет)
        zcr = librosa.feature.zero_crossing_rate(y)
        
        # Агрегируем (среднее и стандартное отклонение)
        # std очень важна: она показывает "стабильность" работы мотора
        vector = np.hstack([
            np.mean(mfccs, axis=1), 
            np.std(mfccs, axis=1),
            np.mean(centroid),
            np.mean(rolloff),
            np.mean(rms),
            np.mean(zcr),
            np.mean(flatness),
            np.percentile(librosa.feature.rms(y=y), 95) # Берем 95-й перцентиль громкости (пики)
        ])
        
        return vector
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [13]:
def extract_winning_features(file_path):
    y, sr = librosa.load(str(file_path), sr=22050)
    
    # 1. Гармоники vs Шум
    y_harm, y_perc = librosa.effects.hpss(y)
    harmonic_ratio = np.mean(y_harm) / (np.mean(y_perc) + 1e-6)
    
    # 2. Мел-спектрограмма
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    # 3. Тембральный профиль
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    tonnetz = librosa.feature.tonnetz(y=y, sr=sr)
    
    # 4. Контраст спектра
    contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr))
    
    return np.hstack([
        np.mean(mfcc, axis=1),
        np.std(mfcc, axis=1),
        np.mean(mel_db, axis=1),
        np.mean(chroma, axis=1),
        np.mean(tonnetz, axis=1),
        harmonic_ratio,
        contrast
    ])

In [15]:
# 2. Загрузка данных и извлечение признаков
df = pd.read_csv(TRAIN_CSV)
print(f"Найдено записей в манифесте: {len(df)}")

# Добавляем session_id и chunk_id для сортировки и окон
df['session_id'] = df['filename'].str.split('_').str[0]
df['chunk_id'] = df['filename'].str.split('_').str[1].str.split('.').str[0].astype(int)
df = df.sort_values(['session_id', 'chunk_id']).reset_index(drop=True)

Найдено записей в манифесте: 570


Функция extract_features_from_audio() берет аудиофайл и вычисляет из него 29 чисел, которые описывают звук:

MFCC (20 коэффициентов) — описывают "форму" звука, тембр
Mel-спектрограмма (16 чисел) — распределение энергии по частотам
Спектральный центроид — "яркость" звука
ZCR — частота пересечения нуля (помогает найти трение/скрежет)
RMS — громкость
Harmonic ratio — соотношение гармоник к шуму
Вместо работы с полным аудиофайлом (тысячи значений) модель работает с этими 29 числами — это делает обучение быстрее и эффективнее.

In [16]:
# извлекаем признаки для каждого аудио файла

X_train = []
for idx in tqdm(range(len(df)), desc="Экстракция признаков"):
    path = DATA_DIR / df.iloc[idx]['filename']
    feat = extract_features_from_audio(path)
    if feat is not None:
        X_train.append(feat)

X_train = np.array(X_train)
print(f"Матрица признаков готова: {X_train.shape}")

# Проверка на NaN в матрице признаков и простая импутация
print('NaNs in X_train:', int(np.isnan(X_train).sum()))
if np.isnan(X_train).any():
    # Быстрое решение: заменить NaN/inf на 0.0 — можно улучшить до медианы
    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    print('Replaced NaN/inf in X_train with 0.0')
    X_train = X_train.astype(np.float32)

Экстракция признаков:   0%|          | 0/570 [00:00<?, ?it/s]

Экстракция признаков: 100%|██████████| 570/570 [10:28<00:00,  1.10s/it]

Матрица признаков готова: (570, 42)
NaNs in X_train: 0


Разделить по блокам для лучшей читаемости

In [17]:
# 3. Нормализация, обучение и сравнение моделей
labels_raw = df['label']
if not np.issubdtype(labels_raw.dtype, np.number):
    label_counts = labels_raw.value_counts()
    print(f"label values: {label_counts.to_dict()}")
    if len(label_counts) == 2:
        # Большая по частоте метка считается нормальной, меньшая — аномальной
        normal_label = label_counts.index[0]
        anomaly_label = label_counts.index[1]
        if label_counts.iloc[1] > label_counts.iloc[0]:
            normal_label, anomaly_label = anomaly_label, normal_label
        label_map = {normal_label: 0, anomaly_label: 1}
        y = labels_raw.map(label_map).astype(int)
    else:
        y = labels_raw.astype('category').cat.codes
else:
    y = labels_raw.astype(int)

# Тест разных contamination значений на валидации
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

best_contamination = 0.02
best_val_roc = 0.0

print("Оптимизация contamination...")
for cont in [0.01, 0.02, 0.05, 0.1]:
    scaler_test = RobustScaler()
    X_train_split_scaled = scaler_test.fit_transform(X_train_split)
    pca_test = PCA(n_components=0.98, svd_solver="full", random_state=42)
    X_train_split_proj = pca_test.fit_transform(X_train_split_scaled)
    
    X_val_scaled = scaler_test.transform(X_val)
    X_val_proj = pca_test.transform(X_val_scaled)
    
    iforest_test = IsolationForest(n_estimators=256, contamination=cont, random_state=42, n_jobs=-1)
    iforest_test.fit(X_train_split_proj)
    val_scores_test = -iforest_test.decision_function(X_val_proj)
    val_roc = roc_auc_score(y_val, val_scores_test)
    print(f"  contamination={cont}: ROC AUC={val_roc:.4f}")
    
    if val_roc > best_val_roc:
        best_val_roc = val_roc
        best_contamination = cont

print(f"Best contamination: {best_contamination}")

contamination = best_contamination
print(f"contamination from labels: {contamination:.4f}")

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
pca = PCA(n_components=0.98, svd_solver="full", random_state=42)
X_proj = pca.fit_transform(X_train_scaled)

iforest = IsolationForest(
    n_estimators=256,
    max_samples="auto",
    contamination=contamination,
    random_state=42,
    n_jobs=-1,
)

iforest.fit(X_proj)

def _anomaly_score(model, X):
    """Вычисляет аномальность, защита от NaN"""
    try:
        if hasattr(model, "decision_function"):
            scores = -model.decision_function(X)
        elif hasattr(model, "score_samples"):
            scores = -model.score_samples(X)
        else:
            scores = np.zeros(X.shape[0])
        # Очистка NaN/inf
        scores = np.nan_to_num(scores, nan=0.0, posinf=1.0, neginf=0.0)
        return scores
    except Exception as e:
        print(f"Error in _anomaly_score: {e}")
        return np.zeros(X.shape[0])

score_vector = _anomaly_score(iforest, X_proj)
score_center = float(np.mean(score_vector))
score_scale = float(np.std(score_vector))
if score_scale < 1e-6:
    score_scale = 1.0

normalized_scores = (score_vector - score_center) / (score_scale + 1e-9)

print(f"IsolationForest ROC AUC (train): {roc_auc_score(y, normalized_scores):.4f}")
print(f"IsolationForest PR AUC (train): {average_precision_score(y, normalized_scores):.4f}")

# Вычислить оптимальные веса на валидации
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_proj,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

iforest.fit(X_train_split)

iforest_scores = _anomaly_score(iforest, X_val)

iforest_auc = roc_auc_score(y_val, iforest_scores)
print(f"IsolationForest validation ROC AUC: {iforest_auc:.4f}")

model_dict = {
    "model": iforest,
}

scaler_dict = {
    "scaler": scaler,
    "pca": pca,
    "score_center": score_center,
    "score_scale": score_scale,
}

joblib.dump(model_dict, MODEL_PATH)
joblib.dump(model_dict, MODEL_PATH.with_name('best_model.pkl'))
joblib.dump(scaler_dict, MODEL_PATH.with_name('scaler.pkl'))
joblib.dump(iforest, MODEL_PATH.with_name('iforest.pkl'))
print(f"Сохранены: best_model.pkl, scaler.pkl, iforest.pkl")

label values: {'normal': 570}
Оптимизация contamination...


c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


  contamination=0.01: ROC AUC=nan


c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


  contamination=0.02: ROC AUC=nan


...

  contamination=0.1: ROC AUC=nan
Best contamination: 0.02
contamination from labels: 0.0200


c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Ensemble ROC AUC (train): nan
Ensemble PR AUC (train): 0.0000


c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Оптимальные веса: IForest=nan, LOF=nan, OCSVM=nan
Сохранены: best_model.pkl, scaler.pkl, iforest.pkl


In [32]:
from sklearn.covariance import EllipticEnvelope


In [18]:
# 4. Валидация на отложенной выборке
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_proj,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Retrain IsolationForest on split
iforest.fit(X_train_split)

val_score_vector = _anomaly_score(iforest, X_val)
val_score_vector = np.nan_to_num(val_score_vector, nan=0.0, posinf=1.0, neginf=0.0)

val_normalized = (val_score_vector - score_center) / (score_scale + 1e-9)
val_normalized = np.nan_to_num(val_normalized, nan=0.0, posinf=1.0, neginf=0.0)

print(f"Validation ROC AUC: {roc_auc_score(y_val, val_normalized):.4f}")
print(f"Validation PR AUC: {average_precision_score(y_val, val_normalized):.4f}")

precision, recall, thresholds = precision_recall_curve(y_val, val_normalized)
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
val_pred = (val_normalized > best_threshold).astype(int)
print(f"Best threshold by F1: {best_threshold:.4f}")
print(classification_report(y_val, val_pred))

Validation ROC AUC: nan
Validation PR AUC: 0.0000
Best threshold by F1: 2.3397
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       114

    accuracy                           1.00       114
   macro avg       1.00      1.00      1.00       114
weighted avg       1.00      1.00      1.00       114



c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\wer28\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
